# Chapter 02: Geometric and Analytical Structures

**Source orientation:** PDF pages 20-25 of *Mathematical Foundations of Geometric Deep Learning*. The source span introduces norms, normed vector spaces, metrics induced by norms, pseudometrics, quasi-metrics, Hausdorff distance, inner products, inner-product-induced norms, the parallelogram law, Gram-Schmidt orthogonalization, and scaled dot-product attention. This notebook uses that span as a map, but all prose, code, diagrams, and checks here are original.

**Chapter goal:** make the hierarchy "metric space -> normed vector space -> inner product space" operational. By the end, each structure should have a visible geometric effect, a small computational interface, and at least one invariant that catches misuse.

**Guiding question:** What extra information appears when a bare set receives distances, when a vector space receives a norm, and when a norm comes from an inner product?

## Translation guide

- **Norm** becomes a function `norm(x)` with three testable promises: zero only at the zero vector, scaling by `alpha` scales by `abs(alpha)`, and a direct route is no longer than a broken route through another vector.
- **Norm-induced metric** becomes `d(x, y) = norm(x - y)`. The same unit ball that measures vector size becomes the shape of distance contours around every point.
- **Pseudometric** keeps the distance calculus but may identify different objects as distance zero. It is useful when the measurement intentionally forgets a coordinate, a phase, or a nuisance transformation.
- **Quasi-metric** in this chapter means a distance-like rule with a relaxed triangle inequality. The examples below use the `p = 1/2` quasi-norm geometry to show why the constant matters.
- **Hausdorff distance** compares two finite sampled shapes by the worst nearest-neighbor miss, not by an average. That makes it sensitive to outliers and missing protrusions.
- **Proximity graph** turns a metric sample into a discrete domain. Radius graphs and k-nearest-neighbor graphs are different answers to the same geometric question: which samples are close enough to communicate?
- **Inner product** adds signed alignment. It induces a norm, supports angles and projections, and makes orthogonal decompositions checkable.
- **Attention scaling** uses an inner product in high dimension. Dividing by `sqrt(d)` keeps random dot-product logits at a stable scale before softmax.

## Visual storyboard and library routing

1. **Norm balls and homogeneity.** Matplotlib is the right durable 2D representation because the shape of the unit ball exposes convexity, anisotropy, and triangle-inequality failure.
2. **Induced metric contours and metric generalizations.** Matplotlib shows translated balls; a small axiom-grid table records which properties survive for a norm metric, a pseudometric, and a quasi-metric.
3. **Hausdorff witnesses.** `scipy.spatial.distance.cdist` computes nearest-neighbor distances, while Matplotlib highlights the two directed worst cases.
4. **Proximity graphs.** `scipy.spatial.KDTree` and NetworkX expose how distance choices become graph edges and connected components.
5. **Projections and Gram-Schmidt.** Matplotlib draws the projection residual; SymPy and NumPy check parallelogram, polarization, orthogonality, and Pythagorean identities.
6. **Attention scaling.** Plotly writes a standalone HTML comparison of raw and scaled dot-product distributions across dimensions, because the learner should be able to toggle dimensions and inspect concentration.

The notebook keeps the computations deliberately small. The point is not performance; the point is to make the chapter's structures visible enough that their invariants feel testable.

In [ ]:
from pathlib import Path
import sys
import json
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial import KDTree
from scipy.spatial.distance import cdist
import networkx as nx
import sympy as sp
import plotly.graph_objects as go
from plotly.subplots import make_subplots

cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents, cwd / "Mathematical-Foundations-of-Geometric-Deep-Learning"]
BOOK_ROOT = None
for candidate in candidates:
    if (candidate / "00-book-index.ipynb").exists() and (candidate / "utils").exists():
        BOOK_ROOT = candidate
        break
if BOOK_ROOT is None:
    raise RuntimeError("Could not find the Mathematical Foundations of GDL book root")

if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import display_artifact, save_matplotlib, save_plotly_html
from utils.notebook_checks import assert_chapter_artifacts, assert_nonblank_image
from utils.plotting import PALETTE, style_axis, close

ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / "chapter-02"
FIGURES = ARTIFACT_ROOT / "figures"
HTML = ARTIFACT_ROOT / "html"
TABLES = ARTIFACT_ROOT / "tables"
CHECKS_DIR = ARTIFACT_ROOT / "checks"
DATA = ARTIFACT_ROOT / "data"
for folder in [FIGURES, HTML, TABLES, CHECKS_DIR, DATA]:
    folder.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(202602)
ARTIFACTS = []
CHECKS = {}

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "#fbfcfe",
    "font.size": 10,
})


def record_artifact(path):
    path = Path(path)
    if path not in ARTIFACTS:
        ARTIFACTS.append(path)
    return path


def jsonable(value):
    if isinstance(value, dict):
        return {str(k): jsonable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [jsonable(v) for v in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.floating, np.integer)):
        return value.item()
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, Path):
        return value.as_posix()
    return value


def save_figure(fig, path, *, dpi=150):
    path = Path(path)
    saved = save_matplotlib(
        fig,
        "chapter-02",
        path.name,
        kind=path.parent.name,
        dpi=dpi,
        root=BOOK_ROOT / "artifacts",
    )
    close(fig)
    return record_artifact(saved)


def save_json(payload, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(jsonable(payload), indent=2, sort_keys=True), encoding="utf-8")
    return record_artifact(path)


def save_csv(frame, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False)
    return record_artifact(path)


def lp_norm(x, p=2):
    x = np.asarray(x, dtype=float)
    if p == np.inf:
        return float(np.max(np.abs(x)))
    return float(np.sum(np.abs(x) ** p) ** (1.0 / p))


def pairwise_lp(points, p=2):
    points = np.asarray(points, dtype=float)
    diff = points[:, None, :] - points[None, :, :]
    if p == np.inf:
        return np.max(np.abs(diff), axis=-1)
    return np.sum(np.abs(diff) ** p, axis=-1) ** (1.0 / p)

BOOK_ROOT

## 1. Norms: the unit ball is the geometry

A norm is more than a formula for a number. Its unit ball tells us how the space treats direction. The Euclidean ball is round; the `L1` ball rewards coordinate-axis moves; the `Linf` ball cares only about the largest coordinate; and the `p = 1/2` example below is intentionally included as a failure mode. It is homogeneous, but its unit "ball" is not convex, and the ordinary triangle inequality breaks.

The inspection target in the next artifact is the boundary shape. Convex unit balls support the norm triangle inequality: if two vectors sit in the unit ball, their average must remain in the unit ball. The `p = 1/2` panel visibly violates that convexity, and the numeric check uses the simplest possible witness, `e1` and `e2`.

In [ ]:
def lp_unit_boundary(p, samples=720):
    theta = np.linspace(0, 2 * np.pi, samples, endpoint=True)
    c, s = np.cos(theta), np.sin(theta)
    if p == np.inf:
        r = 1.0 / np.maximum(np.abs(c), np.abs(s))
    else:
        r = (np.abs(c) ** p + np.abs(s) ** p) ** (-1.0 / p)
    return r * c, r * s

fig, axes = plt.subplots(1, 4, figsize=(13.5, 3.5), sharex=True, sharey=True)
panels = [(1, "L1 norm"), (2, "L2 norm"), (np.inf, "Linf norm"), (0.5, "p = 1/2 quasi-norm")]
for ax, (p, title) in zip(axes, panels):
    x, y = lp_unit_boundary(p)
    fill = PALETTE["gold"] if p == 0.5 else PALETTE["blue"]
    ax.fill(x, y, color=fill, alpha=0.18)
    ax.plot(x, y, color=fill if p == 0.5 else PALETTE["blue"], lw=2)
    ax.axhline(0, color="#8b949e", lw=0.8)
    ax.axvline(0, color="#8b949e", lw=0.8)
    ax.set_xlim(-1.35, 1.35)
    ax.set_ylim(-1.35, 1.35)
    style_axis(ax, title, equal=True)

u = np.array([0.45, 0.28])
alpha = 1.75
axes[1].arrow(0, 0, u[0], u[1], color=PALETTE["red"], width=0.01, length_includes_head=True)
axes[1].arrow(0, 0, alpha * u[0], alpha * u[1], color=PALETTE["green"], width=0.01, length_includes_head=True)
axes[1].text(0.12, 0.11, "u", color=PALETTE["red"])
axes[1].text(0.67, 0.47, "1.75u", color=PALETTE["green"])
axes[3].plot([1, 0, 1], [0, 1, 0], "o--", color=PALETTE["red"], lw=1.5)
axes[3].text(0.20, 0.72, "midpoint leaves\nquasi-ball", color=PALETTE["red"], fontsize=8)
fig.suptitle("Norm balls encode the geometry of size", fontsize=13, color=PALETTE["ink"])
norm_figure = save_figure(fig, FIGURES / "norm-balls-homogeneity.png")

u = np.array([0.6, -1.2])
v = np.array([1.4, 0.35])
alpha = -2.5
rows = []
for p, name in [(1, "L1"), (2, "L2"), (np.inf, "Linf")]:
    homogeneity_residual = abs(lp_norm(alpha * u, p) - abs(alpha) * lp_norm(u, p))
    triangle_slack = lp_norm(u, p) + lp_norm(v, p) - lp_norm(u + v, p)
    zero_test = lp_norm(np.zeros(2), p)
    rows.append({
        "rule": name,
        "homogeneity_residual": homogeneity_residual,
        "triangle_slack": triangle_slack,
        "zero_vector_norm": zero_test,
        "passes_norm_sample": homogeneity_residual < 1e-12 and triangle_slack >= -1e-12 and zero_test == 0.0,
    })

quasi_failure = lp_norm(np.array([1.0, 1.0]), 0.5) - lp_norm(np.array([1.0, 0.0]), 0.5) - lp_norm(np.array([0.0, 1.0]), 0.5)
rows.append({
    "rule": "p=1/2 quasi-norm",
    "homogeneity_residual": abs(lp_norm(alpha * u, 0.5) - abs(alpha) * lp_norm(u, 0.5)),
    "triangle_slack": -quasi_failure,
    "zero_vector_norm": lp_norm(np.zeros(2), 0.5),
    "passes_norm_sample": False,
})

norm_table = pd.DataFrame(rows)
norm_table_path = save_csv(norm_table, TABLES / "norm-axiom-samples.csv")
CHECKS["norms"] = {
    "lp_rules_pass_sample_norm_tests": bool(norm_table.loc[:2, "passes_norm_sample"].all()),
    "quasi_triangle_violation_for_e1_e2": float(quasi_failure),
    "quasi_relaxed_constant_needed_at_least": float(lp_norm([1, 1], 0.5) / (lp_norm([1, 0], 0.5) + lp_norm([0, 1], 0.5))),
}

norm_table

In [ ]:
display_artifact(norm_figure, width=900)
display_artifact(norm_table_path)

## 2. From norms to metrics, and what can go wrong

A normed vector space automatically becomes a metric space by subtracting first and measuring second: `d(u, v) = ||u - v||`. Geometrically, every distance contour around a point is just a translated and rescaled copy of the norm ball. This is why a nearest-neighbor rule can behave differently under `L1`, `L2`, and `Linf` even when the same points are used.

Metrics do not require vector addition, and distance-like rules can weaken different axioms. The pseudometric below forgets the vertical coordinate, so two different points with the same `x` coordinate have distance zero. The quasi-metric-like `p = 1/2` rule keeps non-negativity, identity, and symmetry in this finite test, but it fails the ordinary triangle inequality and only passes a relaxed version with constant `C = 2`.

In [ ]:
center = np.array([0.65, -0.25])
radii = [0.35, 0.70, 1.05]
fig, axes = plt.subplots(1, 3, figsize=(11, 3.6), sharex=True, sharey=True)
for ax, (p, title) in zip(axes, [(1, "L1 distance"), (2, "L2 distance"), (np.inf, "Linf distance")]):
    for radius in radii:
        x, y = lp_unit_boundary(p)
        ax.plot(center[0] + radius * x, center[1] + radius * y, lw=1.8, label=f"r={radius:.2f}")
    ax.scatter([center[0]], [center[1]], color=PALETTE["red"], zorder=4)
    ax.text(center[0] + 0.05, center[1] + 0.05, "v", color=PALETTE["red"])
    style_axis(ax, title, equal=True, xlabel="x1", ylabel="x2")
    ax.set_xlim(-0.75, 1.9)
    ax.set_ylim(-1.55, 1.15)
axes[-1].legend(loc="lower right", fontsize=8)
fig.suptitle("Norm-induced metrics translate the unit ball to every center", fontsize=13, color=PALETTE["ink"])
metric_contours = save_figure(fig, FIGURES / "induced-metric-contours.png")

sample_points = np.array([
    [0.0, 0.0], [0.0, 1.2], [1.0, 0.0], [1.8, 0.9], [-0.4, 0.6]
])

def euclidean_metric(a, b):
    return lp_norm(np.asarray(a) - np.asarray(b), 2)

def x_pseudometric(a, b):
    return abs(float(a[0] - b[0]))

def half_quasi_metric(a, b):
    return lp_norm(np.asarray(a) - np.asarray(b), 0.5)


def axiom_check(distance, points, C=2.0, tol=1e-10):
    n = len(points)
    D = np.array([[distance(points[i], points[j]) for j in range(n)] for i in range(n)])
    nonnegative = bool(np.all(D >= -tol))
    zero_diagonal = bool(np.all(np.abs(np.diag(D)) <= tol))
    identity = bool(zero_diagonal and np.all(D[~np.eye(n, dtype=bool)] > tol))
    symmetry = bool(np.allclose(D, D.T, atol=tol))
    tri = True
    relaxed = True
    worst_ratio = 0.0
    for i in range(n):
        for j in range(n):
            for k in range(n):
                broken = D[i, j] + D[j, k]
                if D[i, k] > broken + tol:
                    tri = False
                if broken > tol:
                    worst_ratio = max(worst_ratio, float(D[i, k] / broken))
                if D[i, k] > C * broken + tol:
                    relaxed = False
    return {
        "nonnegative": nonnegative,
        "zero_iff_same": identity,
        "symmetric": symmetry,
        "triangle": bool(tri),
        "relaxed_C2_triangle": bool(relaxed),
        "worst_triangle_ratio": worst_ratio,
    }

axiom_results = {
    "L2 norm-induced metric": axiom_check(euclidean_metric, sample_points),
    "x-coordinate pseudometric": axiom_check(x_pseudometric, sample_points),
    "p=1/2 quasi-metric sample": axiom_check(half_quasi_metric, sample_points),
}
columns = ["nonnegative", "zero_iff_same", "symmetric", "triangle", "relaxed_C2_triangle"]
rows = list(axiom_results)
grid = np.array([[axiom_results[row][col] for col in columns] for row in rows], dtype=int)
fig, ax = plt.subplots(figsize=(9, 3.2))
ax.imshow(grid, cmap="RdYlGn", vmin=0, vmax=1)
ax.set_xticks(range(len(columns)), [c.replace("_", "\n") for c in columns], fontsize=8)
ax.set_yticks(range(len(rows)), rows, fontsize=9)
for i in range(len(rows)):
    for j in range(len(columns)):
        ax.text(j, i, "yes" if grid[i, j] else "no", ha="center", va="center", color=PALETTE["ink"], fontsize=9)
ax.set_title("Metric axioms separate metrics, pseudometrics, and quasi-metrics", color=PALETTE["ink"])
ax.tick_params(length=0)
for spine in ax.spines.values():
    spine.set_visible(False)
metric_grid = save_figure(fig, FIGURES / "metric-generalizations-axiom-grid.png")
metric_checks_path = save_json(axiom_results, CHECKS_DIR / "metric-generalization-checks.json")

u = np.array([1.2, -0.7])
v = np.array([-0.4, 1.1])
CHECKS["metrics"] = {
    "induced_metric_matches_norm_L1": abs(pairwise_lp(np.array([u, v]), 1)[0, 1] - lp_norm(u - v, 1)) < 1e-12,
    "pseudometric_distinct_zero_pair": x_pseudometric(sample_points[0], sample_points[1]) == 0.0,
    "quasi_triangle_fails": not axiom_results["p=1/2 quasi-metric sample"]["triangle"],
    "quasi_C2_relaxed_passes": axiom_results["p=1/2 quasi-metric sample"]["relaxed_C2_triangle"],
}

pd.DataFrame(axiom_results).T

In [ ]:
display_artifact(metric_contours, width=850)
display_artifact(metric_grid, width=850)
display_artifact(metric_checks_path)

## 3. Hausdorff distance: a worst nearest-neighbor witness

For finite point clouds, the directed Hausdorff term from `A` to `B` is easy to compute: for every point in `A`, find its closest point in `B`, then keep the largest of those closest distances. The symmetric Hausdorff distance is the maximum of the two directions. This is different from a Chamfer-style average, which can hide a single badly missed protrusion.

In the figure, light segments show nearest-neighbor assignments. The two thick segments are the directed witnesses. The larger of those two lengths is the Hausdorff distance. The check records both directed values so the symmetry is visible rather than assumed.

In [ ]:
theta_a = np.linspace(0, 2 * np.pi, 38, endpoint=False)
A = np.column_stack([np.cos(theta_a), 0.55 * np.sin(theta_a)])
angle = 0.25
R = np.array([[np.cos(angle), -np.sin(angle)], [np.sin(angle), np.cos(angle)]])
theta_b = np.linspace(0, 2 * np.pi, 30, endpoint=False)
B_core = np.column_stack([0.78 * np.cos(theta_b), 0.42 * np.sin(theta_b)]) @ R.T + np.array([0.35, 0.10])
B = np.vstack([B_core, [1.85, 0.72]])
D = cdist(A, B)
nearest_B = D.argmin(axis=1)
nearest_A = D.argmin(axis=0)
dir_A_to_B = D.min(axis=1)
dir_B_to_A = D.min(axis=0)
idx_A = int(np.argmax(dir_A_to_B))
idx_B = int(np.argmax(dir_B_to_A))
hausdorff = float(max(dir_A_to_B[idx_A], dir_B_to_A[idx_B]))
chamfer = float(0.5 * (dir_A_to_B.mean() + dir_B_to_A.mean()))

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.4), gridspec_kw={"width_ratios": [1.25, 0.75]})
ax = axes[0]
for i, j in enumerate(nearest_B):
    ax.plot([A[i, 0], B[j, 0]], [A[i, 1], B[j, 1]], color="#b8c2cc", lw=0.5, alpha=0.45)
for j, i in enumerate(nearest_A):
    ax.plot([B[j, 0], A[i, 0]], [B[j, 1], A[i, 1]], color="#d8b55a", lw=0.5, alpha=0.30)
ax.scatter(A[:, 0], A[:, 1], color=PALETTE["blue"], label="A", s=28, zorder=3)
ax.scatter(B[:, 0], B[:, 1], color=PALETTE["gold"], label="B", s=28, zorder=3)
ax.plot([A[idx_A, 0], B[nearest_B[idx_A], 0]], [A[idx_A, 1], B[nearest_B[idx_A], 1]], color=PALETTE["red"], lw=3, label="worst A -> B")
ax.plot([B[idx_B, 0], A[nearest_A[idx_B], 0]], [B[idx_B, 1], A[nearest_A[idx_B], 1]], color=PALETTE["green"], lw=3, label="worst B -> A")
style_axis(ax, "Directed witnesses for finite Hausdorff distance", equal=True)
ax.legend(loc="lower left", fontsize=8)

axes[1].bar(["A -> B", "B -> A", "Hausdorff", "Chamfer avg"], [dir_A_to_B[idx_A], dir_B_to_A[idx_B], hausdorff, chamfer], color=[PALETTE["red"], PALETTE["green"], PALETTE["violet"], PALETTE["gray"]])
style_axis(axes[1], "Worst case vs average", ylabel="distance")
axes[1].tick_params(axis="x", labelrotation=25)
fig.suptitle("Hausdorff distance keeps the largest nearest-neighbor miss", fontsize=13, color=PALETTE["ink"])
hausdorff_figure = save_figure(fig, FIGURES / "hausdorff-directed-witnesses.png")

hausdorff_summary = {
    "directed_A_to_B": float(dir_A_to_B[idx_A]),
    "directed_B_to_A": float(dir_B_to_A[idx_B]),
    "hausdorff": hausdorff,
    "chamfer_average": chamfer,
    "A_witness_index": idx_A,
    "B_witness_index": idx_B,
}
hausdorff_path = save_json(hausdorff_summary, CHECKS_DIR / "hausdorff-summary.json")
CHECKS["hausdorff"] = {
    "hausdorff_is_max_of_directed_terms": abs(hausdorff - max(dir_A_to_B[idx_A], dir_B_to_A[idx_B])) < 1e-12,
    "hausdorff_exceeds_chamfer_average_in_this_example": hausdorff > chamfer,
    "nearest_distance_matrix_shape": list(D.shape),
}

hausdorff_summary

In [ ]:
display_artifact(hausdorff_figure, width=850)
display_artifact(hausdorff_path)

## 4. Proximity graphs: metrics become edges

Geometric deep learning often begins with samples in a metric space and then builds a graph. A radius graph connects pairs whose distance is below a chosen threshold. A k-nearest-neighbor graph gives every point a fixed local budget of neighbors, then usually symmetrizes the result before message passing.

The same coordinates can lead to different connectivity depending on the metric and rule. The figure below keeps the positions fixed and compares a radius graph with a symmetrized k-nearest-neighbor graph. The lab table then varies the metric and radius so that the edge count and component count become inspectable design choices rather than hidden preprocessing details.

In [ ]:
cluster_a = rng.normal(loc=[-0.65, 0.05], scale=[0.20, 0.18], size=(13, 2))
cluster_b = rng.normal(loc=[0.72, 0.12], scale=[0.22, 0.20], size=(13, 2))
bridge = np.array([[-0.12, 0.00], [0.12, 0.03]])
points = np.vstack([cluster_a, bridge, cluster_b])
labels = np.array([0] * len(cluster_a) + [1] * len(bridge) + [2] * len(cluster_b))
positions = {i: points[i] for i in range(len(points))}

radius = 0.34
D2 = pairwise_lp(points, 2)
radius_edges = [(i, j) for i in range(len(points)) for j in range(i + 1, len(points)) if D2[i, j] <= radius]
G_radius = nx.Graph()
G_radius.add_nodes_from(range(len(points)))
G_radius.add_edges_from(radius_edges)

k = 3
tree = KDTree(points)
_, nn_indices = tree.query(points, k=k + 1)
knn_edges = set()
for i, neighbors in enumerate(nn_indices):
    for j in neighbors[1:]:
        knn_edges.add(tuple(sorted((i, int(j)))))
G_knn = nx.Graph()
G_knn.add_nodes_from(range(len(points)))
G_knn.add_edges_from(sorted(knn_edges))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.4), sharex=True, sharey=True)
for ax, graph, title in [(axes[0], G_radius, f"radius graph, r={radius}"), (axes[1], G_knn, f"symmetrized {k}-NN graph")]:
    component_map = {}
    for component_id, component in enumerate(nx.connected_components(graph)):
        for node in component:
            component_map[node] = component_id
    node_colors = [component_map[i] for i in range(len(points))]
    nx.draw_networkx_edges(graph, positions, ax=ax, edge_color="#8b949e", width=1.2, alpha=0.75)
    nx.draw_networkx_nodes(graph, positions, ax=ax, node_color=node_colors, cmap="viridis", node_size=75, edgecolors="white", linewidths=0.8)
    ax.scatter(points[labels == 1, 0], points[labels == 1, 1], facecolors="none", edgecolors=PALETTE["red"], s=180, linewidths=1.6, label="bridge samples")
    style_axis(ax, title, equal=True, xlabel="x", ylabel="y")
    ax.legend(loc="lower right", fontsize=8)
fig.suptitle("A proximity graph is a metric decision made discrete", fontsize=13, color=PALETTE["ink"])
proximity_figure = save_figure(fig, FIGURES / "proximity-graphs-radius-knn.png")

metric_specs = [("L1", 1), ("L2", 2), ("Linf", np.inf)]
radii = [0.28, 0.34, 0.42, 0.52]
lab_rows = []
for metric_name, p in metric_specs:
    Dp = pairwise_lp(points, p)
    for r in radii:
        edges = [(i, j) for i in range(len(points)) for j in range(i + 1, len(points)) if Dp[i, j] <= r]
        G = nx.Graph()
        G.add_nodes_from(range(len(points)))
        G.add_edges_from(edges)
        lab_rows.append({
            "metric": metric_name,
            "radius": r,
            "edge_count": G.number_of_edges(),
            "component_count": nx.number_connected_components(G),
            "largest_component": max(len(c) for c in nx.connected_components(G)),
            "mean_degree": float(np.mean([degree for _, degree in G.degree()])),
        })
proximity_lab = pd.DataFrame(lab_rows)
proximity_table_path = save_csv(proximity_lab, TABLES / "proximity-graph-metric-choice.csv")

adj_radius = nx.to_numpy_array(G_radius, dtype=int)
adj_knn = nx.to_numpy_array(G_knn, dtype=int)
CHECKS["proximity_graphs"] = {
    "radius_graph_edges": G_radius.number_of_edges(),
    "knn_graph_edges": G_knn.number_of_edges(),
    "radius_adjacency_symmetric": bool(np.array_equal(adj_radius, adj_radius.T)),
    "knn_adjacency_symmetric_after_symmetrization": bool(np.array_equal(adj_knn, adj_knn.T)),
    "radius_components": nx.number_connected_components(G_radius),
    "knn_components": nx.number_connected_components(G_knn),
    "lab_rows": len(proximity_lab),
}

proximity_lab

In [ ]:
display_artifact(proximity_figure, width=850)
display_artifact(proximity_table_path)

### Applied lab: choose a graph rule for a message-passing layer

Treat the table as a small preprocessing experiment. Suppose a downstream model needs one connected component but should avoid an almost complete graph. The code filters metric-radius choices by a simple edge budget. This is not an optimization recipe; it is a sanity pattern. Before building layers on top of a graph, check that the geometric rule produced the kind of domain you think it produced.

In [ ]:
node_count = len(points)
max_reasonable_edges = 3 * node_count
connected_sparse_choices = proximity_lab[(proximity_lab["component_count"] == 1) & (proximity_lab["edge_count"] <= max_reasonable_edges)].copy()
connected_sparse_choices = connected_sparse_choices.sort_values(["edge_count", "metric", "radius"])
CHECKS["applied_lab"] = {
    "candidate_count": len(connected_sparse_choices),
    "edge_budget": max_reasonable_edges,
    "has_connected_sparse_choice": len(connected_sparse_choices) > 0,
}
connected_sparse_choices

## 5. Inner products: angles, projections, and orthogonal bases

A norm tells us length. An inner product tells us signed alignment, and from signed alignment we get angles, orthogonality, projections, and orthonormal bases. The hierarchy matters: every inner product induces a norm by `||u|| = sqrt(<u, u>)`, but not every norm comes from an inner product.

The parallelogram law is the diagnostic. The Euclidean norm satisfies it, and the polarization identity recovers the dot product. The `L1` norm fails it, so treating `L1` lengths as if they came with Euclidean angles is a category error. The diagram below uses an ordinary dot product because projections and Gram-Schmidt need exactly that extra structure.

In [ ]:
u = np.array([2.6, 1.4])
v = np.array([1.25, -0.45])
projection = (np.dot(u, v) / np.dot(v, v)) * v
residual = u - projection
cosine_uv = float(np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v)))

basis_vectors = [np.array([2.0, 0.8]), np.array([0.85, 2.15])]
e1 = basis_vectors[0] / np.linalg.norm(basis_vectors[0])
proj_second = np.dot(basis_vectors[1], e1) * e1
orth_second = basis_vectors[1] - proj_second
e2 = orth_second / np.linalg.norm(orth_second)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.7))
ax = axes[0]
for vector, color, label in [(u, PALETTE["blue"], "u"), (v, PALETTE["gray"], "v"), (projection, PALETTE["green"], "proj_v u"), (residual, PALETTE["red"], "orthogonal residual")]:
    origin = np.zeros(2) if label != "orthogonal residual" else projection
    ax.arrow(origin[0], origin[1], vector[0], vector[1], head_width=0.08, head_length=0.10, color=color, length_includes_head=True)
    tip = origin + vector
    ax.text(tip[0] + 0.05, tip[1] + 0.05, label, color=color, fontsize=9)
ax.plot([projection[0], u[0]], [projection[1], u[1]], color=PALETTE["red"], lw=2)
style_axis(ax, "Projection splits u into parallel and orthogonal parts", equal=True, xlabel="x", ylabel="y")
ax.set_xlim(-0.3, 3.1)
ax.set_ylim(-0.8, 2.1)

ax = axes[1]
for vector, color, label in [(basis_vectors[0], PALETTE["blue"], "a1"), (basis_vectors[1], PALETTE["gold"], "a2"), (e1, PALETTE["green"], "e1"), (e2, PALETTE["red"], "e2")]:
    ax.arrow(0, 0, vector[0], vector[1], head_width=0.07, head_length=0.09, color=color, length_includes_head=True)
    ax.text(vector[0] + 0.04, vector[1] + 0.04, label, color=color, fontsize=9)
ax.plot([proj_second[0], basis_vectors[1][0]], [proj_second[1], basis_vectors[1][1]], color=PALETTE["red"], ls="--", lw=1.8)
style_axis(ax, "Gram-Schmidt removes the component already explained", equal=True, xlabel="x", ylabel="y")
ax.set_xlim(-0.3, 2.55)
ax.set_ylim(-0.25, 2.55)
fig.suptitle("Inner products make projections and orthogonal bases computable", fontsize=13, color=PALETTE["ink"])
projection_figure = save_figure(fig, FIGURES / "projection-gram-schmidt-orthogonality.png")

A = np.array([[2.0, 1.0, 1.0], [1.0, 2.0, 0.0], [0.0, 1.0, 2.0]])
columns = [A[:, i] for i in range(A.shape[1])]
Q_cols = []
R = np.zeros((3, 3))
for j, a in enumerate(columns):
    w = a.copy()
    for i, q in enumerate(Q_cols):
        R[i, j] = np.dot(q, a)
        w = w - R[i, j] * q
    R[j, j] = np.linalg.norm(w)
    Q_cols.append(w / R[j, j])
Q = np.column_stack(Q_cols)
reconstruction = Q @ R
orthogonality_residual = float(np.linalg.norm(Q.T @ Q - np.eye(3)))
reconstruction_residual = float(np.linalg.norm(A - reconstruction))
pythagorean_residual = float(abs(np.linalg.norm(Q.sum(axis=1)) ** 2 - sum(np.linalg.norm(Q[:, i]) ** 2 for i in range(3))))

a = sp.Matrix([1, 1])
b = sp.Matrix([1, -1])
l2_norm_sq = lambda z: (z.dot(z))
l1_norm = lambda z: sum(abs(int(value)) for value in z)
parallelogram_l2 = sp.simplify(2 * l2_norm_sq(a) + 2 * l2_norm_sq(b) - l2_norm_sq(a + b) - l2_norm_sq(a - b))
parallelogram_l1 = 2 * l1_norm(a) ** 2 + 2 * l1_norm(b) ** 2 - l1_norm(a + b) ** 2 - l1_norm(a - b) ** 2
x = sp.Matrix([2, -1, 3])
y = sp.Matrix([-1, 4, 2])
polarization_dot = sp.Rational(1, 4) * (l2_norm_sq(x + y) - l2_norm_sq(x - y))

gram_table = pd.DataFrame(Q, columns=["q1", "q2", "q3"])
gram_table_path = save_csv(gram_table, TABLES / "gram-schmidt-orthonormal-basis.csv")
inner_checks = {
    "projection_residual_dot_v": float(np.dot(residual, v)),
    "cosine_u_v": cosine_uv,
    "gram_schmidt_orthogonality_residual": orthogonality_residual,
    "gram_schmidt_reconstruction_residual": reconstruction_residual,
    "pythagorean_residual_for_Q_columns": pythagorean_residual,
    "parallelogram_l2_residual_exact": str(parallelogram_l2),
    "parallelogram_l1_residual_exact": int(parallelogram_l1),
    "polarization_recovers_dot_exact": str(sp.simplify(polarization_dot - x.dot(y))),
}
inner_checks_path = save_json(inner_checks, CHECKS_DIR / "inner-product-gram-schmidt.json")
CHECKS["inner_products"] = {
    "projection_residual_orthogonal_to_v": abs(np.dot(residual, v)) < 1e-12,
    "gram_schmidt_orthonormal": orthogonality_residual < 1e-12,
    "gram_schmidt_reconstructs_A": reconstruction_residual < 1e-12,
    "L2_parallelogram_passes": parallelogram_l2 == 0,
    "L1_parallelogram_fails": parallelogram_l1 != 0,
    "polarization_recovers_dot": sp.simplify(polarization_dot - x.dot(y)) == 0,
}

inner_checks

In [ ]:
display_artifact(projection_figure, width=850)
display_artifact(gram_table_path)
display_artifact(inner_checks_path)

## 6. Attention scaling: dot products in high dimension

The inner product is also a similarity score. In a transformer attention head, query-key logits are dot products. If query and key coordinates behave roughly like independent standard normal variables, the raw dot product has variance proportional to the feature dimension `d`. Larger dimensions therefore create larger logits, which can push softmax toward overly sharp probabilities before learning has done anything meaningful.

Scaled dot-product attention divides by `sqrt(d)`. The interactive artifact compares the raw and scaled distributions. The invariant is statistical rather than exact: raw variance should grow approximately like `d`, while scaled variance should stay near `1`.

In [ ]:
dimensions = [8, 32, 128, 512]
samples = 3500
attention_rows = []
fig = make_subplots(rows=1, cols=2, subplot_titles=("raw q dot k", "q dot k / sqrt(d)"))
colors = [PALETTE["blue"], PALETTE["green"], PALETTE["gold"], PALETTE["red"]]

for dim, color in zip(dimensions, colors):
    q = rng.standard_normal((samples, dim))
    k_vec = rng.standard_normal((samples, dim))
    dots = np.einsum("ij,ij->i", q, k_vec)
    scaled = dots / math.sqrt(dim)
    fig.add_trace(go.Histogram(x=dots, histnorm="probability density", name=f"d={dim}", opacity=0.45, marker_color=color, nbinsx=70), row=1, col=1)
    fig.add_trace(go.Histogram(x=scaled, histnorm="probability density", name=f"d={dim} scaled", opacity=0.45, marker_color=color, nbinsx=70, showlegend=False), row=1, col=2)

    n_queries = 700
    n_keys = 64
    q_batch = rng.standard_normal((n_queries, dim))
    keys = rng.standard_normal((n_queries, n_keys, dim))
    logits = np.einsum("qd,qkd->qk", q_batch, keys)
    logits_scaled = logits / math.sqrt(dim)

    def entropy_from_logits(values):
        shifted = values - values.max(axis=1, keepdims=True)
        probs = np.exp(shifted)
        probs = probs / probs.sum(axis=1, keepdims=True)
        return float(np.mean(-np.sum(probs * np.log(probs + 1e-12), axis=1)))

    attention_rows.append({
        "dimension": dim,
        "raw_variance": float(np.var(dots)),
        "scaled_variance": float(np.var(scaled)),
        "raw_variance_over_d": float(np.var(dots) / dim),
        "raw_softmax_entropy": entropy_from_logits(logits),
        "scaled_softmax_entropy": entropy_from_logits(logits_scaled),
        "uniform_entropy_64_keys": float(math.log(n_keys)),
    })

fig.update_layout(
    title="Scaling by sqrt(d) stabilizes random dot-product logits",
    barmode="overlay",
    template="plotly_white",
    height=480,
    width=980,
)
fig.update_xaxes(title_text="logit value", row=1, col=1)
fig.update_xaxes(title_text="scaled logit value", row=1, col=2)
fig.update_yaxes(title_text="density", row=1, col=1)
attention_html = record_artifact(save_plotly_html(
    fig,
    "chapter-02",
    "attention-scaling-dot-product.html",
    root=BOOK_ROOT / "artifacts",
    include_plotlyjs=True,
    full_html=True,
))

attention_table = pd.DataFrame(attention_rows)
attention_table_path = save_csv(attention_table, TABLES / "attention-scaling-statistics.csv")
attention_summary = {
    "dimensions": dimensions,
    "max_abs_scaled_variance_minus_one": float(np.max(np.abs(attention_table["scaled_variance"].to_numpy() - 1.0))),
    "raw_variance_over_dimension_mean": float(attention_table["raw_variance_over_d"].mean()),
    "scaled_entropy_min": float(attention_table["scaled_softmax_entropy"].min()),
    "raw_entropy_last_dimension": float(attention_table.iloc[-1]["raw_softmax_entropy"]),
    "scaled_entropy_last_dimension": float(attention_table.iloc[-1]["scaled_softmax_entropy"]),
}
attention_checks_path = save_json(attention_summary, CHECKS_DIR / "attention-scaling-summary.json")
CHECKS["attention_scaling"] = {
    "scaled_variance_stays_near_one": attention_summary["max_abs_scaled_variance_minus_one"] < 0.12,
    "raw_variance_tracks_dimension": abs(attention_summary["raw_variance_over_dimension_mean"] - 1.0) < 0.08,
    "scaled_entropy_exceeds_raw_at_largest_dimension": attention_summary["scaled_entropy_last_dimension"] > attention_summary["raw_entropy_last_dimension"],
}

attention_table

In [ ]:
display_artifact(attention_html, width="100%", height=520)
display_artifact(attention_table_path)
display_artifact(attention_checks_path)

## Sanity checks and Takeaways

The checks below are intentionally redundant. They verify that core chapter identities were exercised, every generated artifact exists and is nonempty, static figures are not blank, and the high-dimensional attention experiment produced the expected variance behavior.

The main takeaway is structural: a metric only needs pairwise distances; a norm needs vector subtraction and scaling; an inner product adds angles and orthogonal decompositions. Geometric deep learning moves between these levels constantly, so the notebook records where each level is actually used.

In [ ]:
image_artifacts = [
    norm_figure,
    metric_contours,
    metric_grid,
    hausdorff_figure,
    proximity_figure,
    projection_figure,
]
image_stats = [assert_nonblank_image(path) for path in image_artifacts]
for stat in image_stats:
    stat_path = Path(stat["path"]).resolve()
    stat["path"] = stat_path.relative_to(BOOK_ROOT).as_posix()

required_checks = {
    "norms": CHECKS["norms"]["lp_rules_pass_sample_norm_tests"] and CHECKS["norms"]["quasi_triangle_violation_for_e1_e2"] > 0,
    "metrics": CHECKS["metrics"]["pseudometric_distinct_zero_pair"] and CHECKS["metrics"]["quasi_C2_relaxed_passes"],
    "hausdorff": CHECKS["hausdorff"]["hausdorff_is_max_of_directed_terms"],
    "proximity": CHECKS["proximity_graphs"]["radius_adjacency_symmetric"] and CHECKS["applied_lab"]["has_connected_sparse_choice"],
    "inner_products": all(CHECKS["inner_products"].values()),
    "attention_scaling": all(CHECKS["attention_scaling"].values()),
}
assert all(required_checks.values()), required_checks

final_summary = {
    "source_span": "PDF pages 20-25",
    "artifact_count_before_final_json": len(ARTIFACTS),
    "image_artifact_count": len(image_artifacts),
    "required_checks": required_checks,
    "checks": CHECKS,
    "nonblank_image_stats": image_stats,
}
final_sanity_path = save_json(final_summary, CHECKS_DIR / "final-sanity.json")
records = assert_chapter_artifacts(ARTIFACTS)
print(f"Generated and checked {len(records)} chapter-02 artifacts.")
print(f"Final sanity JSON: {final_sanity_path.relative_to(BOOK_ROOT)}")
records